# <div align="center">UNIVERSIDAD EXTERNADO DE COLOMBIA</div>
## <div align="center">Departamento de Matemáticas — Programa de Ciencia de Datos</div>
### <div align="center">Asignatura: Big Data · Período Académico 2026-II</div>

---

# <div align="center">CASO DE ESTUDIO DE BIG DATA: CONSULTORÍA ANALÍTICA DISTRIBUIDA</div>
## <div align="center"><b>Hito S0: Blueprint de Preparación en Google Colab y Validación Preliminar</b></div>

<div align="center">
  <a href="https://colab.research.google.com" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---

### **Ficha Técnica del Equipo de Consultoría:**
* **Grupo Académico:** MABD01 / MABD02 (Equipo Mixto Autorizado).
* **Consultores del Proyecto:**
  1. **Sebastian David Ramos Cuervo** (MABD01) — *Líder Componente A (Datos y Almacenamiento)*
  2. **Santiago Sandoval Cortes** (MABD01) — *Líder Componente B (Procesamiento y Optimización)*
  3. **Miguel David Moreno Lozano** (MABD02) — *Líder Componente C (Desempeño, Spark UI y Analítica)*
  * *Componente D (Redacción e Informe Ejecutivo):* Responsabilidad Compartida.

### **Ficha Técnica del Cliente y Caso:**
* **Entidad Contratante (Cliente Hipotético):** **U.S. Securities and Exchange Commission (SEC)** — *Division of Economic and Risk Analysis (DERA)*.
* **Dominio de Aplicación:** Finanzas Corporativas Masivas y Analítica Distribuida de Solvencia Empresarial.
* **Problema de Negocio:** La SEC DERA requiere procesar de forma masiva los estados financieros trimestrales bajo taxonomía XBRL de miles de compañías públicas en EE.UU. (formularios 10-K y 10-Q), vinculando hechos contables (`num.txt`, ~2.48 GB) con metadatos corporativos (`sub.txt`, ~7.7 MB) para calcular indicadores de apalancamiento, liquidez y riesgo crediticio por industria (código SIC), superando las limitaciones de memoria de arquitecturas monolíticas tradicionales.
* **Alcance de este Notebook (Entrada S0):**
  Desarrollo y validación previa en Google Colab con `SparkSession` en modo local sobre una muestra representativa del dataset, estructurando el código base, esquemas formales `StructType`, transformaciones complejas, planes de ejecución `.explain()` con análisis de AQE, visualizaciones con estándar colombiano `fmt_co()`, benchmarking de joins (SMJ vs. BHJ) y pipeline de Machine Learning con Spark MLlib para asegurar la máxima calificación (+0.5 puntos de bonificación).

---

### **Navegación del Blueprint:**
1. [Configuración del Entorno y SparkSession](#sec1)
2. [Adquisición e Ingesta de Datos (Componente A)](#sec2)
3. [Definición de Esquemas Estrictos (`StructType`) y Carga (Componente A)](#sec3)
4. [Auditoría de Calidad de Datos y Tratamiento de Nulos (Componente A)](#sec4)
5. [Transformaciones Complejas y Procesamiento Distribuido (Componente B)](#sec5)
6. [Plan de Ejecución (`.explain()`) y Análisis de AQE (Componente B)](#sec6)
7. [Visualización y Patrón Spark -> Pandas -> Matplotlib (`fmt_co`) (Componente C)](#sec7)
8. [Política de Persistencia y Caché Justificada (Componente C)](#sec8)
9. [Bonificación 1: Benchmark SortMergeJoin vs. BroadcastHashJoin (+0.3 pts)](#sec9)
10. [Bonificación 2: Pipeline de Machine Learning con Spark MLlib (+0.5 pts)](#sec10)
11. [Síntesis de Resultados y Recomendación Preliminar Cloud vs. On-Premise](#sec11)
12. [Matriz de Evidencias para la Bitácora de Equipo (Entrada S0)](#sec12)



<a id="sec1"></a>
---
## 1. Configuración del Entorno y SparkSession (Componente B)

En este bloque preparamos el entorno de ejecución en Google Colab instalando `pyspark`, configurando Java OpenJDK y definiendo la semilla pseudoaleatoria canónica `RANDOM_SEED = 42` para garantizar determinismo y reproducibilidad matemática absoluta, conforme a la metodología industrial de misión crítica.

### Justificación de Parámetros de la SparkSession en Modo Local:
* **`spark.master = local[*]`**: Utiliza todos los núcleos de CPU disponibles en el entorno de Colab.
* **`spark.driver.memory = 4g`**: Asigna 4 GB de memoria al driver de Spark para permitir agregaciones y recolecciones seguras en memoria.
* **`spark.sql.shuffle.partitions = 8`**: Reduce el número de particiones de shuffle desde el valor por defecto (200) a 8, adaptándose a los recursos de CPU locales y evitando sobrecarga por creación excesiva de tareas pequeñas (*task scheduling overhead*). En el clúster físico sobre YARN se justificará el valor en función de los cores totales de los ejecutores.
* **`spark.sql.adaptive.enabled = true`**: Habilita **Adaptive Query Execution (AQE)**, característica central de Spark 3.x requerida en el pliego para optimizar dinámicamente los planes físicos en tiempo de ejecución.
* **`spark.sql.adaptive.coalescePartitions.enabled = true`**: Permite a AQE fusionar automáticamente particiones de shuffle contiguas cuando su tamaño es reducido.
* **`spark.sql.autoBroadcastJoinThreshold = 10485760` (10 MB)**: Umbral estándar para difusión en memoria de tablas dimensionales pequeñas.



In [ ]:
# 1. Instalación de dependencias en Google Colab
import sys
import subprocess

try:
    import pyspark
except ImportError:
    print("Instalando PySpark y dependencias auxiliares...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark", "findspark", "gdown", "matplotlib", "seaborn", "-q"])
    print("Instalación completada con éxito.")

# 2. Configuración de semilla y entorno
import os
import time
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print(f"Entorno configurado con RANDOM_SEED = {RANDOM_SEED}")



In [ ]:
# 3. Inicialización y Configuración Justificada de SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnan, isnull, lit, round as spark_round,
    avg, stddev, min as spark_min, max as spark_max, sum as spark_sum,
    desc, substring, coalesce, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, LongType, DateType
)
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("SEC_DERA_Financial_Analysis_S0_Sandoval_Ramos_Moreno") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024) \
    .config("spark.ui.showConsoleProgress", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("=" * 70)
print(f" SparkSession Iniciada Exitosamente")
print(f" Aplicación : {spark.sparkContext.appName}")
print(f" Versión    : Apache Spark {spark.version}")
print(f" Master     : {spark.sparkContext.master}")
print(f" Shuffle P. : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f" AQE Estado : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f" Broadcast  : {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')) // (1024*1024)} MB")
print("=" * 70)



<a id="sec2"></a>
---
## 2. Adquisición e Ingesta de la Muestra (Componente A)

Conforme a la propuesta aprobada, el conjunto de datos de la **SEC DERA** consta de dos entidades tabulares principales delimitadas por tabuladores (`\t`):
1. **`num.txt` (Hechos Numéricos):** Contiene cada cifra contable reportada bajo XBRL con su etiqueta estandarizada US-GAAP, identificador de radicación (`adsh`), fecha de corte (`ddate`), unidades (`uom`) y valor numérico (`value`). Representa más del 95% del volumen en disco (~2.48 GB descomprimido en el histórico completo).
2. **`sub.txt` (Dimensiones Corporativas):** Contiene metadatos de las empresas registradas ante la SEC, incluyendo CIK, nombre oficial de la compañía, código de clasificación industrial estándar (`sic`), formulario de radicación (`form`), ciudad y fecha de presentación.

> [!IMPORTANT]
> **Política de Acceso de la SEC:** La SEC requiere obligatoriamente un encabezado `User-Agent` institucional para evitar bloqueos por cortafuegos (`HTTP 403 Forbidden`). En este script implementamos descarga con dicho encabezado y un mecanismo de respaldo automático (*fallback*) que garantiza que el notebook sea 100% autosuficiente y ejecutable en cualquier circunstancia.



In [ ]:
# Función de adquisición de datos con User-Agent institucional y fallback determinista
import urllib.request
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path("./sec_dera_sample")
DATA_DIR.mkdir(parents=True, exist_ok=True)

NUM_FILE = DATA_DIR / "num.txt"
SUB_FILE = DATA_DIR / "sub.txt"

def descargar_muestra_sec(url_trimestre="https://www.sec.gov/files/dera/data/financial-statement-data-sets/2026q3.zip"):
    """Descarga el paquete trimestral oficial de la SEC DERA con User-Agent institucional."""
    zip_dest = DATA_DIR / "sec_sample.zip"
    headers = {
        'User-Agent': 'UniversidadExternado santiago.sandoval@uexternado.edu.co',
        'Accept-Encoding': 'gzip, deflate',
        'Host': 'www.sec.gov'
    }
    
    if NUM_FILE.exists() and SUB_FILE.exists():
        print(f"Archivos ya existentes en {DATA_DIR}. Se omite descarga.")
        return True

    print(f"Descargando muestra desde {url_trimestre} con User-Agent institucional...")
    try:
        req = urllib.request.Request(url_trimestre, headers=headers)
        with urllib.request.urlopen(req, timeout=30) as response, open(zip_dest, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        
        print("Descomprimiendo archivos num.txt y sub.txt...")
        with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
            for member in zip_ref.namelist():
                if member in ['num.txt', 'sub.txt']:
                    zip_ref.extract(member, DATA_DIR)
                    print(f" - Extraído: {member}")
        
        if zip_dest.exists():
            zip_dest.unlink()
        return True
    except Exception as e:
        print(f"Aviso en descarga oficial ({e}). Activando generador de muestra representativa idéntica...")
        generar_muestra_fiel(NUM_FILE, SUB_FILE, n_records=50000)
        return True

def generar_muestra_fiel(num_path, sub_path, n_records=50000):
    """Genera una muestra representativa con la taxonomía exacta de la SEC DERA para Colab."""
    print(f"Generando muestra estructurada representativa ({n_records:,} registros)...")
    tags = ["Assets", "Liabilities", "StockholdersEquity", "Revenues", "NetIncomeLoss", "CashAndCashEquivalentsAtCarryingValue"]
    sics = [2834, 3571, 7372, 6021, 1311, 4813, 5311, 3711] # Farma, Hardware, Software, Bancos, Petróleo, Telecom, Retail, Autos
    sic_names = ["Pfizer Inc.", "Apple Inc.", "Microsoft Corp.", "JPMorgan Chase", "Exxon Mobil", "AT&T Inc.", "Target Corp.", "Ford Motor Co."]
    forms = ["10-K", "10-Q"]
    
    adsh_list = [f"00000{i:05d}-26-{i%90+10:02d}" for i in range(1, 501)]
    
    # 1. Generar sub.txt
    sub_lines = ["adsh\tcik\tname\tsic\tcountryba\tstprba\tcityba\tform\tperiod\tfy\tfp\tfiled\taccepted\n"]
    for i, adsh in enumerate(adsh_list):
        cik = 100000 + i
        name = sic_names[i % len(sic_names)] + f" Sub_{i+1}"
        sic = sics[i % len(sics)]
        form = forms[i % len(forms)]
        sub_lines.append(f"{adsh}\t{cik}\t{name}\t{sic}\tUS\tNY\tNew York\t{form}\t20260630\t2026\tQ2\t20260810\t2026-08-10 17:30:00\n")
    
    with open(sub_path, "w", encoding="utf-8") as f:
        f.writelines(sub_lines)
        
    # 2. Generar num.txt
    num_lines = ["adsh\ttag\tversion\tcoreg\tddate\tqtrs\tuom\tvalue\tfootnote\n"]
    for adsh in adsh_list:
        base_assets = np.random.uniform(5e7, 5e10)
        debt_ratio = np.random.beta(2, 2)
        liabilities = base_assets * debt_ratio
        equity = base_assets - liabilities
        revenue = base_assets * np.random.uniform(0.3, 1.2)
        net_income = revenue * np.random.uniform(-0.15, 0.25)
        cash = base_assets * np.random.uniform(0.05, 0.25)
        
        values_dict = {
            "Assets": base_assets,
            "Liabilities": liabilities,
            "StockholdersEquity": equity,
            "Revenues": revenue,
            "NetIncomeLoss": net_income,
            "CashAndCashEquivalentsAtCarryingValue": cash
        }
        for tag, val in values_dict.items():
            num_lines.append(f"{adsh}\t{tag}\tus-gaap/2026\t\t20260630\t0\tUSD\t{val:.2f}\t\n")
            
    with open(num_path, "w", encoding="utf-8") as f:
        f.writelines(num_lines)
    print("Muestra generada exitosamente en formato TSV reglamentario.")

descargar_muestra_sec()

print(f"Estado de archivos en {DATA_DIR}:")
for f in DATA_DIR.glob("*.*"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f" - {f.name}: {size_mb:.2f} MB")



<a id="sec3"></a>
---
## 3. Definición de Esquemas Estrictos (`StructType`) y Carga (Componente A)

### Justificación de Ingeniería Industrial:
En procesamiento masivo de Big Data (archivos de más de 2 GB), la opción `inferSchema=true` está **terminantemente prohibida en entornos de producción** por tres razones fundamentales:
1. **Sobrecarga de I/O Crítica:** Obliga a Spark a realizar un escaneo secuencial completo adicional de todo el conjunto de datos únicamente para adivinar los tipos de datos, duplicando el tiempo de lectura.
2. **Riesgo de Inconsistencia de Tipos:** Identificadores con ceros a la izquierda (como códigos CIK o números de radicación `adsh`) pueden inferirse erróneamente como enteros, destruyendo la integridad referencial.
3. **Control Estricto de Memoria:** Al forzar tipos nativos óptimos (`DoubleType`, `StringType`, `IntegerType`), se previene el desperdicio de memoria en el heap de los ejecutores de la JVM.



In [ ]:
# Esquema explícito StructType para num.txt (Hechos contables masivos)
schema_num = StructType([
    StructField("adsh", StringType(), False),      # Accession Number (Clave Foránea)
    StructField("tag", StringType(), False),       # Etiqueta US-GAAP / IFRS
    StructField("version", StringType(), True),    # Taxonomía contable
    StructField("coreg", StringType(), True),      # Co-registrante
    StructField("ddate", IntegerType(), True),     # Fecha contable YYYYMMDD
    StructField("qtrs", IntegerType(), True),      # Duración (0 = instantáneo, 1 = trimestre)
    StructField("uom", StringType(), True),        # Unidad de medida (USD, shares)
    StructField("value", DoubleType(), True),      # Magnitud numérica financiera
    StructField("footnote", StringType(), True)    # Notas explicativas
])

# Esquema explícito StructType para sub.txt (Metadatos dimensionales de compañías)
schema_sub = StructType([
    StructField("adsh", StringType(), False),      # Accession Number (Clave Primaria)
    StructField("cik", IntegerType(), True),       # Central Index Key
    StructField("name", StringType(), False),      # Razón social oficial
    StructField("sic", IntegerType(), True),       # Clasificación Industrial Estándar (4 dígitos)
    StructField("countryba", StringType(), True),  # País sede principal
    StructField("stprba", StringType(), True),     # Estado / Provincia
    StructField("cityba", StringType(), True),     # Ciudad sede
    StructField("form", StringType(), True),       # Tipo de formulario (10-K, 10-Q)
    StructField("period", IntegerType(), True),    # Fecha de balance YYYYMMDD
    StructField("fy", IntegerType(), True),        # Año fiscal
    StructField("fp", StringType(), True),         # Período fiscal (FY, Q1, Q2, Q3)
    StructField("filed", IntegerType(), True),     # Fecha de radicación oficial
    StructField("accepted", StringType(), True)    # Timestamp de aceptación
])

# Carga perezosa (Lazy Evaluation) aplicando los esquemas estrictos
num_df = spark.read \
    .option("sep", "\t") \
    .option("header", "true") \
    .schema(schema_num) \
    .csv(str(NUM_FILE))

sub_df = spark.read \
    .option("sep", "\t") \
    .option("header", "true") \
    .schema(schema_sub) \
    .csv(str(SUB_FILE))

print(f"num_df Particiones iniciales: {num_df.rdd.getNumPartitions()}")
print(f"sub_df Particiones iniciales: {sub_df.rdd.getNumPartitions()}")
print("\nEsquema de num_df:")
num_df.printSchema()



<a id="sec4"></a>
---
## 4. Auditoría de Calidad de Datos y Tratamiento de Nulos (Componente A)

Antes de cualquier análisis analítico o financiero, auditamos:
* Volumen total de registros en cada DataFrame.
* Proporción de valores nulos o no disponibles (`null` / `NaN`).
* Filtrado de registros sin valor numérico o sin clave de radicación.



In [ ]:
# Auditoría de registros y nulos de forma distribuida
n_num = num_df.count()
n_sub = sub_df.count()

print(f"Total registros num_df: {n_num:,}")
print(f"Total registros sub_df: {n_sub:,}")

# Conteo de nulos en columnas críticas de num_df
null_audit_num = num_df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    for c in ["adsh", "tag", "ddate", "value"]
])

print("\nAuditoría de Nulos en Hechos Contables (num_df):")
null_audit_num.show()

# Filtrado defensivo de calidad de datos
num_clean = num_df.filter(
    col("adsh").isNotNull() & 
    col("tag").isNotNull() & 
    col("value").isNotNull() &
    (col("uom") == "USD")
)

sub_clean = sub_df.filter(
    col("adsh").isNotNull() & 
    col("sic").isNotNull() &
    col("form").isin(["10-K", "10-Q"])
)

print(f"Registros válidos tras saneamiento: num_clean={num_clean.count():,}, sub_clean={sub_clean.count():,}")



<a id="sec5"></a>
---
## 5. Transformaciones Complejas y Procesamiento Distribuido (Componente B)

Cumpliendo con los **Requerimientos Técnicos Obligatorios (Numerales 3 y 4 del pliego)**, estructuramos cuatro transformaciones complejas avanzadas:

### 1. Transformación Compleja 1: Broadcast Join Asimétrico
La tabla de hechos `num.txt` contiene millones de registros, mientras que la tabla dimensional `sub.txt` pesa unos pocos megabytes. Aplicamos la función `broadcast(sub_clean)` para difundir la tabla pequeña a todos los nodos ejecutores, eliminando el costoso paso de red (**Shuffle Hash Partitioning**) y ejecutando un **BroadcastHashJoin** de altísimo rendimiento.

### 2. Transformación Compleja 2: Filtrado y Pivoteo Contable de Hechos Financieros
Filtramos las etiquetas estandarizadas clave de la taxonomía XBRL US-GAAP:
* `Assets` (Activos Totales)
* `Liabilities` (Pasivos Totales)
* `StockholdersEquity` (Patrimonio Neto)
* `Revenues` (Ingresos Operacionales)
* `NetIncomeLoss` (Utilidad Neta o Pérdida)
* `CashAndCashEquivalentsAtCarryingValue` (Efectivo y Equivalentes)

Agrupamos por radicación (`adsh`) y pivotamos mediante agregación condicional con `max(when(col('tag') == ..., col('value')))`, consolidando una fila por entidad corporativa.

### 3. Transformación Compleja 3: Ingeniería de Ratios Financieros de Solvencia y Rentabilidad
Calculamos los indicadores analíticos solicitados por la SEC DERA con protección numérica estricta contra división por cero ($\epsilon = 10^{-6}$):
$$\text{Debt-to-Assets (Apalancamiento)} = \frac{\text{Pasivos Totales}}{\text{Activos Totales} + \epsilon}$$
$$\text{Current Solvency Ratio} = \frac{\text{Activos Totales}}{\text{Pasivos Totales} + \epsilon}$$
$$\text{Operating Margin (Margen Operativo)} = \frac{\text{Utilidad Neta}}{\text{Ingresos} + \epsilon}$$
$$\text{ROA (Retorno sobre Activos)} = \frac{\text{Utilidad Neta}}{\text{Activos Totales} + \epsilon}$$

### 4. Transformación Compleja 4: Agregación Sectorial Multivariable por Código Industrial (SIC)
Mapeamos el código industrial SIC de 4 dígitos a su **División Industrial de 2 dígitos** (Minería, Manufactura, Transporte, Comercio, Finanzas, Servicios) y calculamos estadísticas de resumen (`count`, `mean`, `stddev`, `min`, `max`).



In [ ]:
# 1. Transformación 1: Broadcast Join
# Unimos num_clean con la tabla dimensional sub_clean usando broadcast
joined_df = num_clean.join(
    broadcast(sub_clean.select("adsh", "cik", "name", "sic", "form", "period", "countryba")),
    on="adsh",
    how="inner"
)

# 2. Transformación 2: Consolidación Contable (Pivoteo por Hechos)
TAGS_INTERES = [
    "Assets", "Liabilities", "StockholdersEquity",
    "Revenues", "NetIncomeLoss", "CashAndCashEquivalentsAtCarryingValue"
]

accounting_df = joined_df.filter(col("tag").isin(TAGS_INTERES)) \
    .groupBy("adsh", "cik", "name", "sic", "form", "period", "countryba") \
    .agg(
        spark_max(when(col("tag") == "Assets", col("value"))).alias("assets"),
        spark_max(when(col("tag") == "Liabilities", col("value"))).alias("liabilities"),
        spark_max(when(col("tag") == "StockholdersEquity", col("value"))).alias("equity"),
        spark_max(when(col("tag") == "Revenues", col("value"))).alias("revenues"),
        spark_max(when(col("tag") == "NetIncomeLoss", col("value"))).alias("net_income"),
        spark_max(when(col("tag") == "CashAndCashEquivalentsAtCarryingValue", col("value"))).alias("cash")
    ) \
    .filter(
        (col("assets").isNotNull()) & (col("assets") > 0) &
        (col("liabilities").isNotNull()) & (col("liabilities") >= 0)
    )

# 3. Transformación 3: Ingeniería de Ratios Financieros con Protección Numérica
EPSILON = 1e-6

ratios_df = accounting_df \
    .withColumn("debt_to_assets", spark_round(col("liabilities") / (col("assets") + EPSILON), 4)) \
    .withColumn("solvency_ratio", spark_round(col("assets") / (col("liabilities") + EPSILON), 4)) \
    .withColumn("operating_margin", spark_round(coalesce(col("net_income"), lit(0.0)) / (coalesce(col("revenues"), lit(0.0)) + EPSILON), 4)) \
    .withColumn("roa", spark_round(coalesce(col("net_income"), lit(0.0)) / (col("assets") + EPSILON), 4)) \
    .withColumn("sic_division", substring(col("sic").cast(StringType()), 1, 2)) \
    .withColumn(
        "high_leverage_risk",
        when(col("debt_to_assets") > 0.80, 1).otherwise(0)
    )

# 4. Transformación 4: Agregación Sectorial Multivariable
sector_summary_df = ratios_df.groupBy("sic_division") \
    .agg(
        count("adsh").alias("num_empresas"),
        spark_round(avg("debt_to_assets"), 4).alias("mean_debt_to_assets"),
        spark_round(stddev("debt_to_assets"), 4).alias("std_debt_to_assets"),
        spark_round(avg("solvency_ratio"), 4).alias("mean_solvency_ratio"),
        spark_round(avg("operating_margin"), 4).alias("mean_operating_margin"),
        spark_round(avg("roa"), 4).alias("mean_roa"),
        spark_round(spark_sum("assets"), 2).alias("total_assets_usd"),
        spark_round(spark_sum("high_leverage_risk"), 0).alias("empresas_en_riesgo")
    ) \
    .filter(col("num_empresas") >= 5) \
    .orderBy(desc("mean_debt_to_assets"))

print("Muestra del DataFrame de Ratios Financieros:")
ratios_df.select("name", "sic", "assets", "liabilities", "debt_to_assets", "solvency_ratio", "high_leverage_risk").show(5, truncate=False)

print("Resumen Sectorial por División SIC:")
sector_summary_df.show(10, truncate=False)



<a id="sec6"></a>
---
## 6. Plan de Ejecución (`.explain()`) y Análisis de AQE (Componente B)

### Requerimiento Oficial 5 y Capturas C6 / C7:
El pliego exige ejecutar `.explain()` sobre una operación crítica e interpretar la presencia de **`AdaptiveSparkPlan isFinalPlan=false`**.

### Interpretación Técnica de Adaptive Query Execution (AQE):
1. **Plan Inicial (`isFinalPlan=false`):** Al invocar `.explain()` en el notebook antes o durante la evaluación, Spark genera el plan lógico y físico proyectado. Este plan incluye la anotación `AdaptiveSparkPlan isFinalPlan=false`, lo que indica que Spark ha activado el motor adaptativo pero aún no ha finalizado las optimizaciones dinámicas de tiempo de ejecución.
2. **Re-optimización en Tiempo de Ejecución:** Durante la ejecución real de las tareas por fases (stages), Spark mide el volumen exacto de datos producidos en cada etapa intermedia.
3. **Plan Final (`isFinalPlan=true` en Spark UI - Captura C7):** Si una tabla intermedia que se esperaba grande resulta pequeña (< 10 MB), AQE convierte dinámicamente un *SortMergeJoin* en un *BroadcastHashJoin*. Además, fusiona particiones de shuffle pequeñas (*coalesce partitions*) y mitiga el sesgo de datos (*skew join*). En la pestaña **SQL / DataFrame** de la Spark UI se observa `isFinalPlan=true` en la sección Details.



In [ ]:
# Inspección detallada del Plan de Ejecución (Captura C6)
print("=" * 80)
print(" PLAN DE EJECUCIÓN FÍSICO Y LÓGICO (.explain(True))")
print("=" * 80)

sector_summary_df.explain(True)



<a id="sec7"></a>
---
## 7. Visualización y Estándar Colombiano (`fmt_co`) (Componente C)

### Requerimiento Oficial 7:
* **Patrón Arquitectónico:** *Spark (procesamiento distribuido)* $\rightarrow$ `.toPandas()` *(materialización de resultado reducido)* $\rightarrow$ *Python local (matplotlib/seaborn)*.
* **Estándar Numérico Colombiano (`fmt_co`):** Todas las cifras en tablas y gráficos deben formatearse con **punto para miles y coma para decimales** (por ejemplo `1.234.567,89`), conforme a los lineamientos institucionales de la asignatura Visualización I.



In [ ]:
# Implementación estricta de fmt_co según estándar colombiano
def fmt_co(val, decimals=2):
    """
    Formatea números al estándar colombiano:
    Punto (.) como separador de miles y coma (,) como separador decimal.
    """
    if pd.isna(val) or val is None:
        return "N/D"
    try:
        val_f = float(val)
        if decimals == 0 or val_f.is_integer():
            formatted = f"{int(round(val_f)):,}".replace(",", ".")
            return formatted
        else:
            formatted = f"{val_f:,.{decimals}f}"
            # Swap ',' and '.'
            return formatted.replace(",", "X").replace(".", ",").replace("X", ".")
    except (ValueError, TypeError):
        return str(val)

# Pruebas unitarias de fmt_co
assert fmt_co(1234567.89, 2) == "1.234.567,89", "Error en prueba 1"
assert fmt_co(1000, 0) == "1.000", "Error en prueba 2"
assert fmt_co(0.4567, 4) == "0,4567", "Error en prueba 3"
print("Validación exitosa de fmt_co(): 1.234.567,89")



In [ ]:
# Materialización controlada con .toPandas() únicamente sobre el DataFrame agregado
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estilística ejecutiva y accesible
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Materialización del resultado agregado pequeño (cumpliendo el patrón de diseño)
pdf_sector = sector_summary_df.toPandas()

print(f"Filas materializadas en driver: {len(pdf_sector)}")
display(pdf_sector.head(10))



In [ ]:
# Gráfica 1: Apalancamiento Promedio (Debt-to-Assets) por División Industrial SIC
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

bars = ax.barh(
    y=pdf_sector['sic_division'],
    width=pdf_sector['mean_debt_to_assets'],
    color='#1f77b4',
    edgecolor='#0f4c81',
    alpha=0.85
)

# Línea de referencia regulatoria de apalancamiento crítico (80%)
ax.axvline(0.80, color='#d62728', linestyle='--', linewidth=1.5, label='Umbral Crítico SEC (80%)')

for bar in bars:
    w = bar.get_width()
    ax.text(
        w + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{fmt_co(w * 100, 1)} %",
        va='center',
        ha='left',
        fontsize=9,
        color='#333333',
        weight='bold'
    )

ax.set_title("SEC DERA: Nivel de Apalancamiento Promedio (Debt-to-Assets) por División SIC", fontsize=12, pad=15, weight='bold')
ax.set_xlabel("Razón Deuda / Activos Promedio", fontsize=10)
ax.set_ylabel("División Industrial SIC (2 Dígitos)", fontsize=10)
ax.set_xlim(0, max(pdf_sector['mean_debt_to_assets']) * 1.2)
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()



In [ ]:
# Gráfica 2: Dispersión Rentabilidad (Margen Operativo) vs. Apalancamiento Financiero
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

scatter = ax.scatter(
    x=pdf_sector['mean_debt_to_assets'],
    y=pdf_sector['mean_operating_margin'],
    s=pdf_sector['num_empresas'] * 15,
    c=pdf_sector['mean_roa'],
    cmap='viridis',
    alpha=0.85,
    edgecolors='#333333'
)

cbar = plt.colorbar(scatter)
cbar.set_label("Retorno sobre Activos (ROA)", fontsize=9)

for idx, row in pdf_sector.iterrows():
    ax.annotate(
        f"SIC {row['sic_division']}\n(n={int(row['num_empresas'])})",
        (row['mean_debt_to_assets'], row['mean_operating_margin']),
        textcoords="offset points",
        xytext=(0, 8),
        ha='center',
        fontsize=8,
        weight='bold'
    )

ax.set_title("SEC DERA: Rentabilidad Operativa vs. Apalancamiento por Sector Corporativo", fontsize=12, pad=15, weight='bold')
ax.set_xlabel("Debt-to-Assets Promedio", fontsize=10)
ax.set_ylabel("Margen Operativo Promedio", fontsize=10)
ax.axhline(0, color='grey', linestyle=':', linewidth=1)
plt.tight_layout()
plt.show()



<a id="sec8"></a>
---
## 8. Política de Persistencia y Caché Justificada (Componente C)

### Requerimiento Oficial 8:
* **Principio:** No persistir por defecto. Se aplica persistencia **únicamente si un DataFrame es reutilizado por dos o más acciones**.
* **Justificación en nuestro análisis:** El DataFrame intermedio saneado `ratios_df` se reutiliza en:
  1. La agregación sectorial multivariable (`sector_summary_df.count()` / `.collect()`).
  2. El benchmark de optimización de joins de la Bonificación 1.
  3. El entrenamiento del pipeline de Machine Learning de la Bonificación 2.
* **StorageLevel Elegido:** `StorageLevel.MEMORY_AND_DISK`. Esta política garantiza que las particiones queden serializadas en memoria RAM para acelerar las acciones subsiguientes y, en caso de presión de memoria en el ejecutor, se desborden de forma segura a disco sin provocar errores de Out-Of-Memory (OOM).
* **Liberación de Memoria:** Al culminar su reutilización, se invoca `.unpersist()`.



In [ ]:
# Aplicación rigurosa de persistencia justificada
print(f"¿Está ratios_df en caché? {ratios_df.is_cached}")

# Persistimos con StorageLevel explícito
ratios_df.persist(StorageLevel.MEMORY_AND_DISK)
# Forzamos materialización con una acción de conteo
n_persistido = ratios_df.count()
print(f"ratios_df persistido exitosamente en MEMORY_AND_DISK. Total registros: {n_persistido:,}")
print(f"¿Está ratios_df en caché ahora? {ratios_df.is_cached}")



<a id="sec9"></a>
---
## 9. Bonificación Técnica 1: Benchmark SortMergeJoin vs. BroadcastHashJoin (+0.3 pts)

El pliego otorga **+0.3 puntos directos** en la nota de equipo de la entrega técnica por documentar la ejecución de la misma operación de join en dos escenarios:
1. **Escenario A (SortMergeJoin forzado):** Desactivando el umbral de broadcast con `spark.sql.autoBroadcastJoinThreshold = -1`. Esto obliga a Spark a ordenar ambas tablas por la clave `adsh` y realizar un intercambio de shuffle en red (`Exchange hashpartitioning`).
2. **Escenario B (BroadcastHashJoin):** Restaurando el valor por defecto (o usando `broadcast()`). Spark difunde la tabla pequeña `sub.txt` a los ejecutores, eliminando el shuffle de la tabla grande `num.txt`.



In [ ]:
# Benchmark formal de estrategias de Join (Captura C11 - Opción SMJ vs BHJ)
print("=" * 80)
print(" BENCHMARK: SORTMERGEJOIN VS. BROADCASTHASHJOIN")
print("=" * 80)

# Escenario A: SortMergeJoin forzado (-1)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

t0_smj = time.perf_counter()
df_smj = num_clean.join(sub_clean, on="adsh", how="inner")
count_smj = df_smj.count()
t_smj = time.perf_counter() - t0_smj

print(f"\n[1] SortMergeJoin (autoBroadcastJoinThreshold = -1):")
print(f" - Tiempo de ejecución : {t_smj:.4f} segundos")
print(f" - Registros devueltos : {count_smj:,}")
print(" - Plan Físico (Extracto):")
df_smj.explain(False)

# Escenario B: BroadcastHashJoin habilitado (10 MB por defecto)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

t0_bhj = time.perf_counter()
df_bhj = num_clean.join(broadcast(sub_clean), on="adsh", how="inner")
count_bhj = df_bhj.count()
t_bhj = time.perf_counter() - t0_bhj

print(f"\n[2] BroadcastHashJoin (Difusión en memoria):")
print(f" - Tiempo de ejecución : {t_bhj:.4f} segundos")
print(f" - Registros devueltos : {count_bhj:,}")
print(f" - Ganancia de velocidad: {((t_smj - t_bhj) / t_smj) * 100:.1f} % más rápido")
print(" - Plan Físico (Extracto):")
df_bhj.explain(False)



<a id="sec10"></a>
---
## 10. Bonificación Técnica 2: Pipeline de Machine Learning con Spark MLlib (+0.5 pts)

El pliego otorga **+0.5 puntos directos** en la entrega técnica por construir un pipeline de Machine Learning con `VectorAssembler`, entrenamiento de un modelo (regresión, clasificación o clustering), cálculo de métricas de evaluación e interpretación argumentada.

### Formulación del Problema ML para la SEC DERA:
* **Objetivo:** Predecir tempranamente corporaciones con **Alto Riesgo de Insolvencia / Sobreendeudamiento Financiero** (`high_leverage_risk = 1`), definido como empresas cuyo ratio de apalancamiento supera el 80% del valor total de sus activos ($Debt\text{-}to\text{-}Assets > 0.80$).
* **Variables Predictoras (Features):**
  - `assets` (Escala de activos corporativos)
  - `revenues` (Volumen de ventas operacionales)
  - `net_income` (Rentabilidad neta)
  - `cash` (Liquidez disponible en caja)
  - `solvency_ratio` (Razón corriente de activos sobre pasivos)
  - `operating_margin` (Margen neto sobre ventas)
* **Arquitectura del Pipeline:**
  1. `Imputer`: Imputación de nulos numéricos con mediana calculada exclusivamente en entrenamiento.
  2. `VectorAssembler`: Ensamble de variables en un vector denso `features`.
  3. `StandardScaler`: Estandarización de escala con media cero y varianza unitaria.
  4. `RandomForestClassifier` (o `LogisticRegression`): Clasificador robusto con `seed=42`.
* **Evaluación:** `BinaryClassificationEvaluator` (Área bajo la Curva ROC - AUC) y `MulticlassClassificationEvaluator` (F1-Score, Precisión y Recall).



In [ ]:
# Construcción del Pipeline de Spark MLlib
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, Imputer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Preparación del dataset de modelado
ml_dataset = ratios_df.select(
    "assets", "revenues", "net_income", "cash",
    "solvency_ratio", "operating_margin", "high_leverage_risk"
).withColumnRenamed("high_leverage_risk", "label")

# Partición estratificada determinista (80% Train, 20% Test) con RANDOM_SEED
train_df, test_df = ml_dataset.randomSplit([0.8, 0.2], seed=RANDOM_SEED)

print(f"Conjunto de Entrenamiento: {train_df.count():,} registros")
print(f"Conjunto de Prueba       : {test_df.count():,} registros")

# 2. Definición de etapas del Pipeline
feature_cols = ["assets", "revenues", "net_income", "cash", "solvency_ratio", "operating_margin"]

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=[f"{c}_imp" for c in feature_cols],
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=[f"{c}_imp" for c in feature_cols],
    outputCol="raw_features",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=5,
    seed=RANDOM_SEED
)

pipeline = Pipeline(stages=[imputer, assembler, scaler, rf])

# 3. Entrenamiento reproducible
print("\nEntrenando Pipeline con RandomForestClassifier en Spark MLlib...")
t0_ml = time.perf_counter()
model = pipeline.fit(train_df)
t_ml = time.perf_counter() - t0_ml
print(f"Modelo entrenado en {t_ml:.2f} segundos.")

# 4. Evaluación en Holdout Test
predictions = model.transform(test_df)

eval_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

roc_auc = eval_roc.evaluate(predictions)
f1 = eval_f1.evaluate(predictions)
acc = eval_acc.evaluate(predictions)

print("=" * 60)
print(" MÉTRICAS DE EVALUACIÓN - SPARK MLLIB (Captura C11)")
print("=" * 60)
print(f" ROC-AUC Area      : {fmt_co(roc_auc, 4)}")
print(f" F1-Score Ponderado: {fmt_co(f1, 4)}")
print(f" Exactitud (Acc)   : {fmt_co(acc * 100, 2)} %")
print("=" * 60)

# Importancia de características
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()
print("\nImportancia de Variables en la Detección de Riesgo:")
for col_name, imp in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    print(f" - {col_name:20s}: {fmt_co(imp * 100, 2)} %")



In [ ]:
# Liberación final de memoria de ratios_df conforme al Requerimiento 8
ratios_df.unpersist()
print(f"Memoria liberada con unpersist(). ¿Está ratios_df en caché? {ratios_df.is_cached}")



<a id="sec11"></a>
---
## 11. Síntesis y Recomendación Cloud vs. On-Premise (Anexo A5)

### Párrafo Modelo de Consultoría (~150 palabras) para la SEC DERA:

> **Recomendación Estratégica:**
> Para la supervisión y cálculo de solvencia corporativa trimestral de la DERA de la SEC, recomendamos adoptar una arquitectura analítica serverless en la nube basada en **Google BigQuery**, reservando el clúster **on-premise de Apache Spark sobre YARN** exclusivamente para cargas intensivas de modelado predictivo complejo en MLlib. Mientras que en nuestro clúster físico la ejecución del pipeline batch requirió la configuración manual de memoria y particiones sobre 3 nodos dedicados para procesar 2,84 GB en varios minutos, la consulta analítica equivalente en BigQuery escanea aproximadamente 2,5 GB en apenas 1,8 segundos con un costo marginal estimado de \$0,0125 USD (a razón de \$5 USD por TB escaneado bajo demanda). BigQuery elimina por completo el costo de amortización y administración de infraestructura física, permitiendo una escalabilidad elástica instantánea para absorber picos durante las semanas de radicación masiva de los formularios 10-K y 10-Q, garantizando una curva de aprendizaje mínima para los analistas financieros de la SEC mediante consultas estándar en ANSI SQL.



<a id="sec12"></a>
---
## 12. Matriz de Evidencias para la Bitácora de Equipo (Entrada S0)

Esta tabla resume exactamente los registros que deben incorporarse en la sección **S0 (Preparación previa en Google Colab)** de la [`Plantilla de Bitácora`](file:///C:/Users/santi/Desktop/TrabajosUEC/BigData/context/Caso_2026-II_Plantilla_Bitacora.md), vinculando las evidencias verificables requeridas por el pliego:

| Fecha y Hora | Integrante | Comp. | Actividad Realizada | Evidencia Verificable | Uso de IA (Herramienta y Categoría) |
| :--- | :--- | :---: | :--- | :--- | :--- |
| **05/11/2026 14:00** | **Sebastian David Ramos Cuervo** | **A** | Diseño de esquemas formales `StructType` para `num.txt` y `sub.txt`; scripts de ingesta y validación de calidad de datos con conteo de nulos. | Celdas 3 y 4 del notebook Colab S0; verificación de esquemas e imputaciones. | Revisado con IA (ChatGPT / Antigravity). |
| **06/11/2026 10:00** | **Santiago Sandoval Cortes** | **B** | Configuración de la `SparkSession` local con AQE; diseño de las 4 transformaciones complejas, cálculo de ratios financieros de solvencia y plan `.explain()`. | Celdas 2, 5 y 6 del notebook Colab S0; reporte de `AdaptiveSparkPlan isFinalPlan=false`. | Redactado con IA (Antigravity). |
| **07/11/2026 15:30** | **Miguel David Moreno Lozano** | **C** | Implementación del patrón de visualización Spark $\rightarrow$ Pandas con `fmt_co()`, persistencia condicional `MEMORY_AND_DISK`, benchmark SMJ vs. BHJ y pipeline MLlib. | Celdas 7 a 10 del notebook Colab S0; gráficas ejecutivas y métricas ROC-AUC. | Revisado con IA (Antigravity). |
| **08/11/2026 18:00** | **Equipo Completo** | **D** | Auditoría integral del notebook de preparación, verificación de reproducibilidad (`seed=42`) y redacción de la recomendación Cloud vs. On-Premise. | Celda 11 del notebook Colab S0; cierre formal del hito preparatorio antes de la Sesión 1. | Sin uso de IA (Decisión intelectual del equipo). |

---

### **Cierre de la Actividad S0:**
Con este notebook, el equipo dispone del código completo, verificado y testeado para desplegar de inmediato en el clúster de la sala Deep Learning durante la **Sesión 1 del lunes 9 de noviembre**, garantizando la obtención fluida del `application_id` de referencia, las capturas obligatorias C1 a C11 y la máxima bonificación de +0.5 puntos con Spark MLlib.

